# Tools Calling: The Action Engine

In Agentic AI, **Tools** are the primary interface between reasoned logic and the physical world. This module covers the end-to-end lifecycle of function calling — from atomic interface design and schema validation to stateful LangGraph execution and production best practices.

In [7]:
import os
import json
from typing import List, Optional, Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool, StructuredTool
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage

load_dotenv()

model = "gemini-2.5-flash"
llm = ChatGoogleGenerativeAI(model=model, temperature=0)

print(f"Tools environment initialized with model: {model}")

Tools environment initialized with model: gemini-2.5-flash


## 1. Atomic Tool Design

The **Single Responsibility Principle** is the golden rule. A tool should perform exactly one task with surgical precision. The docstring is the only documentation the model receives — it must be specific about *when* and *how* to use the tool.

In [8]:
@tool
def get_stock_price(ticker: str) -> float:
    """Fetches the real-time trading price for a given stock ticker.
    Input must be an uppercase ticker symbol (e.g., 'AAPL', 'GOOGL').
    Do NOT use this for revenue, earnings, or company news."""
    prices = {"AAPL": 185.92, "GOOGL": 142.33, "MSFT": 398.22}
    return prices.get(ticker.upper(), 0.0)

@tool
def get_company_news(company_name: str) -> str:
    """Fetches the latest headline for a company.
    Use this ONLY for news queries, NOT for stock prices or financials."""
    return f"Latest headline for {company_name}: Q4 earnings beat expectations."

print(f"Tool 1: {get_stock_price.name} -> {get_stock_price.description[:60]}...")
print(f"Tool 2: {get_company_news.name} -> {get_company_news.description[:60]}...")

Tool 1: get_stock_price -> Fetches the real-time trading price for a given stock ticker...
Tool 2: get_company_news -> Fetches the latest headline for a company.
    Use this ONLY...


## 2. Structured Argument Validation (Pydantic)

For tools requiring multiple arguments, **Pydantic** schemas enforce types, constraints, and provide detailed field-level descriptions. This prevents the LLM from hallucinating incorrect data types or missing required fields.

In [9]:
class RevenueSearchArgs(BaseModel):
    ticker: str = Field(description="Stock ticker in uppercase (e.g., AAPL).")
    year: int = Field(description="Fiscal year for the report in YYYY format.")

def fetch_revenue(ticker: str, year: int) -> str:
    data = {"AAPL": {2023: "383B", 2022: "394B"}, "GOOGL": {2023: "307B"}}
    result = data.get(ticker, {}).get(year, "Not available")
    return f"Revenue for {ticker} in {year}: ${result}"

revenue_tool = StructuredTool.from_function(
    func=fetch_revenue,
    name="get_historical_revenue",
    description="Retrieves a company's annual revenue for a specific fiscal year.",
    args_schema=RevenueSearchArgs
)

print(f"Tool: {revenue_tool.name}")
print(f"Schema: {revenue_tool.args}")

Tool: get_historical_revenue
Schema: {'ticker': {'description': 'Stock ticker in uppercase (e.g., AAPL).', 'title': 'Ticker', 'type': 'string'}, 'year': {'description': 'Fiscal year for the report in YYYY format.', 'title': 'Year', 'type': 'integer'}}


## 3. Tool Binding & Schema Inspection

Binding tools to an LLM converts Python functions into a structured JSON schema. This schema is transmitted to the model API, telling it what capabilities are available during its reasoning turn.

In [10]:
tools = [get_stock_price, get_company_news, revenue_tool]
llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke("What was Apple's revenue in 2023?")

if response.tool_calls:
    for call in response.tool_calls:
        print(f"Model selected: {call['name']} with args {call['args']}")
else:
    print(f"Model answered directly: {response.content[:100]}")

Model selected: get_historical_revenue with args {'year': 2023.0, 'ticker': 'AAPL'}


## 4. Parallel vs. Sequential Execution

Gemini models can request **multiple tool calls in a single turn**. Parallel execution is efficient for independent data fetches. Sequential execution is required when Tool B depends on the output of Tool A.

In [11]:
parallel_query = "Get the current stock price for AAPL and also get the current stock price for MSFT"
parallel_response = llm_with_tools.invoke(parallel_query)

print(f"Parallel calls requested: {len(parallel_response.tool_calls)}")
for call in parallel_response.tool_calls:
    print(f"  -> {call['name']}({call['args']})")

Parallel calls requested: 2
  -> get_stock_price({'ticker': 'AAPL'})
  -> get_stock_price({'ticker': 'MSFT'})


## 5. The Message-Execution Cycle

Understanding the low-level lifecycle is critical for debugging:
1. **LLM returns** an `AIMessage` containing a `tool_calls` list.
2. **Application executes** each tool call against the actual Python function.
3. **Results are returned** as `ToolMessage` objects with matching `tool_call_id`.
4. **LLM receives** the observations and formulates the final answer.

In [12]:
# Step 1: Send query - LLM decides which tool to call
user_query = "What is the price of AAPL?"
human_msg = HumanMessage(content=user_query)
ai_msg = llm_with_tools.invoke(user_query)
print(f"Step 1 - AI requested: {ai_msg.tool_calls}")

# Step 2: Execute each tool call and collect ToolMessages
tool_results = []
for tc in ai_msg.tool_calls:
    result = get_stock_price.invoke(tc['args'])
    tool_results.append(ToolMessage(content=str(result), tool_call_id=tc['id']))
    print(f"Step 2 - Executed: {tc['name']} -> {result}")

# Step 3: Feed the FULL conversation back (Human + AI + ToolMessages)
final = llm_with_tools.invoke([human_msg, ai_msg] + tool_results)
print(f"Step 3 - Final Answer: {final.content}")

Step 1 - AI requested: [{'name': 'get_stock_price', 'args': {'ticker': 'AAPL'}, 'id': 'aba9ed45-ed92-4a73-aa24-b018fa218921', 'type': 'tool_call'}]
Step 2 - Executed: get_stock_price -> 185.92
Step 3 - Final Answer: The price of AAPL is 185.92.


## 6. Observation-Driven Error Recovery

A resilient agent treats errors as **Observations**, not crashes. If an API returns a 404 or a timeout, return that error string to the model so it can reason about the failure and attempt a different strategy.

In [13]:
@tool
def safe_api_call(endpoint: str) -> str:
    """Calls an external API endpoint. Returns the result or a descriptive error."""
    try:
        if "internal" in endpoint:
            raise ConnectionError("Timeout after 30s")
        if "restricted" in endpoint:
            raise PermissionError("Error 403: Access Denied")
        return f"API Response from {endpoint}: 200 OK"
    except Exception as e:
        return f"TOOL_ERROR: {str(e)}. Suggest trying a different endpoint."

print(safe_api_call.invoke("public/users"))
print(safe_api_call.invoke("internal/admin"))

API Response from public/users: 200 OK
TOOL_ERROR: Timeout after 30s. Suggest trying a different endpoint.


## 7. Dynamic Toolsets & Selection Logic

In systems with hundreds of tools, binding all of them to every turn wastes tokens and confuses the model. **Dynamic Toolsets** filter and bind only the tools relevant to the current intent.

In [14]:
TOOL_REGISTRY = {
    "finance": [get_stock_price, revenue_tool],
    "news": [get_company_news],
    "ops": [safe_api_call]
}

def bind_tools_for_intent(intent: str):
    selected = TOOL_REGISTRY.get(intent, [])
    print(f"Intent: '{intent}' -> Binding {len(selected)} tools: {[t.name for t in selected]}")
    return llm.bind_tools(selected)

finance_llm = bind_tools_for_intent("finance")
news_llm = bind_tools_for_intent("news")

Intent: 'finance' -> Binding 2 tools: ['get_stock_price', 'get_historical_revenue']
Intent: 'news' -> Binding 1 tools: ['get_company_news']


## 8. LangGraph Integration: The ToolNode

**LangGraph** formalizes tool execution into a directed graph. The `ToolNode` is a specialized node that automatically executes tool calls found in the state and routes observations back to the LLM node.

In [15]:
from langgraph.prebuilt import ToolNode

graph_tools = [get_stock_price, get_company_news, safe_api_call]
tool_node = ToolNode(tools=graph_tools)

print(f"ToolNode initialized with {len(graph_tools)} tools:")
for t in graph_tools:
    print(f"  - {t.name}")

ToolNode initialized with 3 tools:
  - get_stock_price
  - get_company_news
  - safe_api_call


## 9. Human-in-the-loop (Security & Breakpoints)

Critical actions — executing trades, sending emails, deleting records — should never be fully autonomous. LangGraph enables **graph interrupts**, pausing execution before a tool node to await human approval.

In [16]:
# LangGraph interrupt configuration:
#
# workflow = StateGraph(AgentState)
# workflow.add_node("agent", call_model)
# workflow.add_node("action", tool_node)
# workflow.add_edge("agent", "action")
# workflow.add_edge("action", "agent")
#
# app = workflow.compile(interrupt_before=["action"])
#
# The graph pauses BEFORE executing any tool, awaiting human signal.

print("LangGraph HITL: compile(interrupt_before=['action'])")
print("The graph pauses before tool execution, awaiting human approval.")

LangGraph HITL: compile(interrupt_before=['action'])
The graph pauses before tool execution, awaiting human approval.


## 10. Observability & Performance Tracing

In multi-tool systems, debugging latency and accuracy is paramount. **LangSmith** enables per-tool tracing, capturing call duration, raw arguments, and output for every execution.

In [17]:
# LangSmith integration (requires LANGCHAIN_API_KEY)
# export LANGCHAIN_TRACING_V2=true
# export LANGCHAIN_API_KEY=your_key

metrics = {
    "Tool Recall": "How often the model picks the correct tool",
    "Avg Latency": "Mean execution time per tool (ms)",
    "Error Rate": "Percentage of tool calls returning errors",
    "Token Efficiency": "Average tokens consumed per tool observation"
}

print("Production Observability Metrics:")
for metric, desc in metrics.items():
    print(f"  {metric}: {desc}")

Production Observability Metrics:
  Tool Recall: How often the model picks the correct tool
  Avg Latency: Mean execution time per tool (ms)
  Error Rate: Percentage of tool calls returning errors
  Token Efficiency: Average tokens consumed per tool observation


## 11. Top 10 Pro Tips for Tool Engineering

The following expert principles, each with a concrete code example, form the definitive checklist for building production-grade tool interfaces.

### Tip 1: Descriptions are Prompts
The docstring is the model's only guidance. Be explicit about when to use — and when NOT to use — the tool.

In [18]:
@tool
def search_bad(q: str) -> str:
    """Searches for stuff."""
    return f"Result for {q}"

@tool
def search_sec_filings(company_name: str) -> str:
    """Searches SEC EDGAR for the latest 10-K annual filing.
    Use ONLY for regulatory filings. Do NOT use for stock prices or news."""
    return f"10-K filing for {company_name}: Revenue $50B (Simulated)"

print(f"Bad:  '{search_bad.description}'")
print(f"Good: '{search_sec_filings.description}'")

Bad:  'Searches for stuff.'
Good: 'Searches SEC EDGAR for the latest 10-K annual filing.
    Use ONLY for regulatory filings. Do NOT use for stock prices or news.'


### Tip 2: Type-Safe Schemas
Use Pydantic `BaseModel` for multi-argument tools to enforce types and prevent hallucinated parameters.

In [19]:
class KPIArgs(BaseModel):
    start_date: str = Field(description="Start date in ISO format (YYYY-MM-DD).")
    end_date: str = Field(description="End date in ISO format (YYYY-MM-DD).")
    metric: Literal["revenue", "profit", "margin"] = Field(description="KPI to fetch.")

def fetch_kpi(start_date: str, end_date: str, metric: str) -> str:
    return f"{metric.capitalize()} from {start_date} to {end_date}: $42M (Simulated)"

kpi_tool = StructuredTool.from_function(
    func=fetch_kpi, name="fetch_kpi", args_schema=KPIArgs,
    description="Fetches a KPI metric for a date range."
)
print(f"Enforced schema: {kpi_tool.args}")

Enforced schema: {'start_date': {'description': 'Start date in ISO format (YYYY-MM-DD).', 'title': 'Start Date', 'type': 'string'}, 'end_date': {'description': 'End date in ISO format (YYYY-MM-DD).', 'title': 'End Date', 'type': 'string'}, 'metric': {'description': 'KPI to fetch.', 'enum': ['revenue', 'profit', 'margin'], 'title': 'Metric', 'type': 'string'}}


### Tip 3: Atomic Design
One tool, one job. Separate read and write operations into distinct tools.

In [20]:
@tool
def read_record(record_id: str) -> str:
    """Reads a single record by its unique ID. Read-only operation."""
    return f"Record {record_id}: name=Acme, status=active"

@tool
def archive_record(record_id: str) -> str:
    """Archives a record. This is a WRITE operation that changes status to 'archived'."""
    return f"Record {record_id} archived successfully."

print(f"Read:  {read_record.name}")
print(f"Write: {archive_record.name}")

Read:  read_record
Write: archive_record


### Tip 4: Error as Observation
Catch all exceptions internally and return them as strings. The model treats errors as observations and adapts.

In [21]:
@tool
def resilient_lookup(query: str) -> str:
    """Performs a lookup that may fail. Returns error details on failure."""
    try:
        if not query.strip():
            raise ValueError("Empty query provided")
        return f"Found 3 results for '{query}'"
    except Exception as e:
        return f"LOOKUP_ERROR: {str(e)}. Try rephrasing the query."

print(resilient_lookup.invoke("valid search"))
print(resilient_lookup.invoke(""))

Found 3 results for 'valid search'
LOOKUP_ERROR: Empty query provided. Try rephrasing the query.


### Tip 5: Side-Effect Transparency
Clearly warn about destructive or irreversible operations in the docstring.

In [22]:
@tool
def drop_table(table_name: str) -> str:
    """PERMANENTLY drops a database table and all its data.
    WARNING: This action is IRREVERSIBLE. Requires explicit user confirmation."""
    return f"Table '{table_name}' dropped permanently."

print(drop_table.description)

PERMANENTLY drops a database table and all its data.


### Tip 6: Input Normalization
Handle common LLM variability (whitespace, casing) inside the tool to prevent silent failures.

In [23]:
@tool
def lookup_ticker(company_name: str) -> str:
    """Looks up the stock ticker for a company by name."""
    normalized = company_name.strip().lower()
    tickers = {"apple": "AAPL", "google": "GOOGL", "microsoft": "MSFT"}
    return tickers.get(normalized, f"Ticker not found for: '{company_name}'")

print(lookup_ticker.invoke("  Apple  "))
print(lookup_ticker.invoke("GOOGLE"))
print(lookup_ticker.invoke("microsoft"))

AAPL
GOOGL
MSFT


### Tip 7: Enum Constraints
Use `Literal` types to restrict model choices to a fixed set of valid options.

In [24]:
class TradeOrder(BaseModel):
    ticker: str = Field(description="Stock ticker in uppercase.")
    action: Literal["buy", "sell", "hold"] = Field(description="Trade action.")
    quantity: int = Field(description="Number of shares.", gt=0)

def place_trade(ticker: str, action: str, quantity: int) -> str:
    return f"ORDER: {action.upper()} {quantity}x {ticker}"

trade_tool = StructuredTool.from_function(
    func=place_trade, name="place_trade", args_schema=TradeOrder,
    description="Places a stock trade. Action restricted to buy, sell, or hold."
)
print(f"Constrained args: {trade_tool.args}")

Constrained args: {'ticker': {'description': 'Stock ticker in uppercase.', 'title': 'Ticker', 'type': 'string'}, 'action': {'description': 'Trade action.', 'enum': ['buy', 'sell', 'hold'], 'title': 'Action', 'type': 'string'}, 'quantity': {'description': 'Number of shares.', 'exclusiveMinimum': 0, 'title': 'Quantity', 'type': 'integer'}}


### Tip 8: Token-Efficient Outputs
Prune API responses to only the fields the model needs. Verbose outputs waste context window.

In [25]:
@tool
def get_company_profile(ticker: str) -> str:
    """Returns a concise company profile with only essential fields."""
    full = {"ticker": ticker, "name": "Apple Inc.", "sector": "Technology",
            "price": 185.92, "pe_ratio": 28.5, "internal_id": "X9F2",
            "last_audit": "2024-01-15", "risk_code": "A2", "employees": 164000}
    essential = {k: full[k] for k in ["ticker", "name", "sector", "price", "pe_ratio"]}
    return json.dumps(essential)

print(f"Pruned output: {get_company_profile.invoke('AAPL')}")

Pruned output: {"ticker": "AAPL", "name": "Apple Inc.", "sector": "Technology", "price": 185.92, "pe_ratio": 28.5}


### Tip 9: In-Description Examples
For complex input formats, embed few-shot examples directly in the docstring.

In [26]:
@tool
def run_analytics_query(filter_expr: str) -> str:
    """Runs a structured query against the analytics database.

    Format: field:operator:value

    Examples:
        - "revenue:gt:1000000" (revenue greater than 1M)
        - "status:eq:active" (status equals active)
        - "date:gte:2024-01-01" (date on or after Jan 1, 2024)
    """
    return f"Query '{filter_expr}': 42 records matched."

print(run_analytics_query.invoke("revenue:gt:1000000"))

Query 'revenue:gt:1000000': 42 records matched.


### Tip 10: Consistent Naming
Never rename parameters between versions. Add new fields with defaults for backward compatibility.

In [27]:
class PriceArgsV1(BaseModel):
    ticker: str = Field(description="Stock ticker symbol.")

# BAD v2: renaming 'ticker' to 'symbol' breaks existing agent reasoning
# class PriceArgsV2Bad(BaseModel):
#     symbol: str = Field(description="Stock ticker symbol.")  # BREAKING!

# GOOD v2: new optional field, original field preserved
class PriceArgsV2(BaseModel):
    ticker: str = Field(description="Stock ticker symbol.")
    include_history: bool = Field(default=False, description="Include 30-day price history.")

print(f"V1: {list(PriceArgsV1.model_fields.keys())}")
print(f"V2 (compatible): {list(PriceArgsV2.model_fields.keys())}")

V1: ['ticker']
V2 (compatible): ['ticker', 'include_history']


---

## Conclusion

Tools transform an LLM from a static predictor into a dynamic problem solver. By mastering atomic interface design, the message-execution lifecycle, stateful LangGraph orchestration, and the 10 professional engineering principles covered in this module, developers can build agents that interact with external systems reliably and safely.